# Лабораторная 1 – Гусев Иван

In [23]:
import pandas as pd
from nltk.corpus import stopwords
import pymorphy3


morph = pymorphy3.MorphAnalyzer()

### Объединяем .csv

In [24]:
negative = pd.read_csv("negative.csv", header=None, sep=';')
positive = pd.read_csv("positive.csv", header=None, sep=';')
dataset = pd.concat([negative, positive])
dataset.head()

,0,1,2,3,4,5,6,7,8,9,10,11
0,408906762813579264,1386325944,dugarchikbellko,на работе был полный пиддес :| и так каждое за...,-1,0,0,0,8064,111,94,2
1,408906818262687744,1386325957,nugemycejela,"Коллеги сидят рубятся в Urban terror, а я из-з...",-1,0,0,0,26,42,39,0
2,408906858515398656,1386325966,4post21,@elina_4post как говорят обещаного три года жд...,-1,0,0,0,718,49,249,0
3,408906914437685248,1386325980,Poliwake,"Желаю хорошего полёта и удачной посадки,я буду...",-1,0,0,0,10628,207,200,0
4,408906914723295232,1386325980,capyvixowe,"Обновил за каким-то лешим surf, теперь не рабо...",-1,0,0,0,35,17,34,0


### Удаляем стоп-слова

In [25]:
stop_words = set(stopwords.words("russian"))
stop_words.remove("хорошо")
print(stop_words)

{'они', 'еще', 'на', 'для', 'тогда', 'где', 'что', 'ты', 'никогда', 'без', 'тоже', 'под', 'с', 'три', 'и', 'себя', 'сейчас', 'этой', 'по', 'иногда', 'про', 'ну', 'будто', 'один', 'он', 'вдруг', 'всех', 'перед', 'более', 'эти', 'было', 'уж', 'нет', 'через', 'много', 'всю', 'к', 'лучше', 'а', 'если', 'ей', 'бы', 'со', 'всегда', 'после', 'раз', 'об', 'у', 'был', 'их', 'всего', 'меня', 'опять', 'куда', 'больше', 'чего', 'моя', 'она', 'в', 'разве', 'ему', 'какая', 'ни', 'ж', 'свою', 'такой', 'им', 'я', 'быть', 'при', 'почти', 'эту', 'за', 'будет', 'но', 'вот', 'этом', 'вас', 'него', 'из', 'вам', 'конечно', 'надо', 'здесь', 'этого', 'над', 'другой', 'уже', 'нельзя', 'есть', 'себе', 'там', 'тот', 'чуть', 'может', 'между', 'же', 'кто', 'до', 'теперь', 'ничего', 'впрочем', 'зачем', 'мне', 'нее', 'том', 'ведь', 'ее', 'хоть', 'его', 'мы', 'сам', 'чтоб', 'нас', 'ли', 'наконец', 'какой', 'не', 'о', 'ним', 'них', 'только', 'потом', 'да', 'нибудь', 'того', 'как', 'или', 'ней', 'вы', 'были', 'два', 'п

In [30]:
def text_preprocess(text: str) -> str:
    """
    Функция предобработки текст: приведение к нижнему регистру, токенизация, лемматизация
    """
    normalized_text = text
    normalized_text = normalized_text.lower()
    normalized_text = [word for word in normalized_text.split() if word not in stop_words]
    normalized_text = [morph.parse(word)[0].normal_form for word in normalized_text]
    return " ".join(normalized_text)

In [ ]:
# Применяем предобработку текста ко всему датасету
dataset.iloc[:, 3] = dataset.iloc[:, 3].apply(text_preprocess)
dataset = dataset.iloc[:, 3]

In [37]:
# Считываем LinisCrowd 2015
sentiment_dataset = pd.read_csv("words_all_full_rating.csv", encoding="cp1251", sep=";", index_col="Words")
sentiment_dataset.head()

,mean,dispersion,average rate,Unnamed: 4
Words,,,,
аборигенный,"-0,25","0,433012701892219",0,NaN
аборт,-1,"0,816496580927726",-1,NaN
абрамович,0,0,0,NaN
абсолютный,"0,333333333333333","0,471404520791032",0,NaN
абстрактный,"-0,111111111111111","0,87488976377909",0,NaN


In [ ]:
def sentiment_analysis(text: str, sentiment_df: pd.DataFrame) -> list:
    """
    Функция, которая преобразует текст в вектор из 
    среднего, максимального, минимального, 
    суммарного значения и количества положительных и отрицательных значений
    """
    text = text.split()